In [ ]:
# Import libraries
import rasterio as rio
from rasterio.mask import mask
from rasterio.plot import plotting_extent
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import os
import warnings
from scipy.stats import gamma

In [2]:
gdf_test = gpd.read_file('Muestras/muestras_tp4_gk5_wgs84_bosque_1.shp')

In [3]:
gdf_test

,Id,nombre,geometry
0,8,forestacion3,"POLYGON ((5591916.822 6247768.838, 5592436.6 6..."


In [ ]:
TIF_PATH = ''
BACK_VAL = 0
SAMPLES_PATH = ''
SAMPLES_COL = ''
LIST_PROBS = [50, 90, 95]

In [ ]:
def read_check_vector(tif_path, samples_path):
    """
    Read and check whether the CRS of a vector samples file matches the CRS of a
    raster file. If they don't match, reproject the samples to the
    raster's CRS and emit a warning.

    Parameters
    ----------
    tif_path : str or Path
        Path to the raster (.tif) file.
    samples_path : str or Path
        Path to the vector samples file (e.g., shapefile, geojson).

    Returns
    -------
    geopandas.GeoDataFrame
        The samples GeoDataFrame, reprojected to match the raster's CRS
        if needed.

    Raises
    ------
    ValueError
        If either the raster or the samples file has no CRS defined.
    """
    gdf_samples = gpd.read_file(samples_path)

    with rio.open(tif_path) as src:
        raster_crs = src.crs

    # Guard against missing CRS on either side
    if raster_crs is None:
        raise ValueError(f"Raster '{tif_path}' has no CRS defined.")
    if gdf_samples.crs is None:
        raise ValueError(f"Samples file '{samples_path}' has no CRS defined.")

    if gdf_samples.crs != raster_crs:
        warnings.warn(
            f"CRS mismatch: samples CRS ({gdf_samples.crs}) does not match "
            f"raster CRS ({raster_crs}). Reprojecting samples to match raster.",
            UserWarning,
            stacklevel=2,
        )
        gdf_samples = gdf_samples.to_crs(raster_crs)

    return gdf_samples
        
        
def clip_tif(tif_path, samples_path, samples_col, back_val=0):
    """
    Clip a raster with each polygon in a vector samples file and extract
    the pixel values within each polygon, excluding background values.

    For each polygon in the samples file, the raster is clipped to that
    polygon's extent, the specified background value is masked out (set
    to NaN and removed), and the remaining valid pixel values are stored
    in a dictionary keyed by the polygon's identifier.

    Parameters
    ----------
    tif_path : str or Path
        Path to the raster (.tif) file to clip.
    samples_path : str or Path
        Path to the vector samples file (e.g., shapefile, geojson)
        containing the polygons to clip with.
    samples_col : str
        Name of the column in the samples file to use as the identifier
        (dictionary key) for each polygon's extracted values.
    back_val : int or float, optional
        Background/nodata value to exclude from the extracted pixel
        values (default is 0).

    Returns
    -------
    dict
        Dictionary mapping each polygon's identifier (from `samples_col`)
        to a 1D numpy array of valid (non-background, non-NaN) pixel
        values extracted from that polygon.
    """
    with rio.open(tif_path) as src:
        gdf_samples = read_check_vector(tif_path, samples_path)

        dict_samp_val = {}
        for idx, row in gdf_samples.iterrows():
            geom = [row.geometry] # shapes takes a list of geometries
            out_image, out_transform = mask(src, geom, crop=True)

            x = out_image
            x[0][x[0] == back_val] = np.nan
            x = x[~np.isnan(x)]

            dict_samp_val[row[samples_col]] = x

    return dict_samp_val

def error_vs_probability(enl, minError_dB=0.1, maxError_dB=4, stepError_dB=0.01):
    """
    For a given ENL (l), compute probability for a range of error_dB values.

    Parameters
    ----------
    enl : float
        Equivalent Number of Looks (ENL), shape parameter of the Gamma pdf.
    minError_dB, maxError_dB, stepError_dB : float
        Range of error_dB values to evaluate.

    Returns
    -------
    error_dBs : ndarray
        The error_dB values evaluated.
    probabilities : ndarray
        The corresponding probability for each error_dB.
    """
    error_dBs = np.arange(minError_dB, maxError_dB + 1e-9, stepError_dB)

    lower = 10 ** (-error_dBs / 10)
    upper = 10 ** (error_dBs / 10)

    probabilities = gamma.cdf(upper, a=l, scale=1.0 / enl) - gamma.cdf(lower, a=enl, scale=1.0 / l)

    return error_dBs, probabilities

def get_unct(enl, p):
    """
    Get the error bound (in dB) corresponding to a target probability,
    for a given ENL, by nearest-neighbor lookup.

    Computes the probability curve for the given ENL over a range of
    error_dB values (via `error_vs_probability`), then finds the
    error_dB value whose associated probability is closest to the
    target probability `p`.

    Parameters
    ----------
    enl : float
        Equivalent Number of Looks (ENL), shape parameter of the
        underlying Gamma distribution.
    p : float
        Target probability (between 0 and 1) for which to find the
        closest matching error_dB.

    Returns
    -------
    error_dB
        The error_dB value whose corresponding probability is nearest
        to `p`.
    prob_ear
        Real nearest probability to the defined by the user
    """
    error_dBs, probabilities = error_vs_probability(enl)

    idx = np.abs(probabilities - p).argmin() # Get the index of nearest probability value (shortest absolute distance)
    prob_near = probabilities[idx]
    error_dB = error_dBs[idx]
    

    return error_dB, prob_near
        

def get_dict_stats(tif_path, samples_path, samples_col):
    """
    Compute per-sample statistics (mean, standard deviation, and ENL)
    from raster values clipped by each polygon in a samples file.

    For each sample polygon, extracts the corresponding raster pixel
    values (via `clip_tif`), then computes the mean, standard
    deviation, and Equivalent Number of Looks (ENL = (mean/std)^2)
    for that sample.

    Parameters
    ----------
    tif_path : str or Path
        Path to the raster (.tif) file.
    samples_path : str or Path
        Path to the vector samples file (e.g., shapefile, geojson).
    samples_col : str
        Name of the column in the samples file used to identify each
        polygon/sample.

    Returns
    -------
    pandas.DataFrame
        DataFrame with one row per sample, containing columns:
        'Sample', 'Mean', 'Std', and 'ENL'.

    """
    
    dict_samp_val = clip_tif(tif_path, samples_path, samples_col)

    list_samp = []
    list_mean = []
    list_std = []
    list_enl = []

    for samp in dict_samp_val.keys():
        vals = dict_samp_val[samp]
        mean = np.nanmean(vals)
        std = np.nanstd(vals)
        enl = (mean / std) ** 2

        list_samp.append(samp)
        list_mean.append(mean)
        list_std.append(std)
        list_enl.append(enl)
        
    dict_samp_stats = pd.DataFrame({
        'Sample': list_samp,
        'Mean': list_mean,
        'Std': list_std,
        'ENL': list_enl,
    })

    return dict_samp_stats
        
    
def get_dict_ru(dict_samp_stats, p):
    
    #dict_samp_stats = get_dict_stats(tif_path, samples_path, samples_col)
    
    list_incrad = []
    list_realprob = []
    
    for idx, row in dict_samp_stats.iterrows():
        
        error_dB, prob_near = get_unct(enl, p/100)
        
        list_incrad.append(error_dB)
        list_realprob.append(prob_near)
        
    
    dict_samp_stats[f'Rad. Unc. {p} [dB]'] = list_incrad
    dict_samp_stats[f'Real. Prob. [%]'] = list_realprob
    
    dict_samp_ru = dict_samp_stats
    
    return dict_samp_ru


def RadUnc(tif_path, samples_path, samples_col, list_probs, back_val=0, output='output'):
    
    dict_samp_stats = get_dict_stats(tif_path, samples_path, samples_col)
    
    for p in list_probs:
        
        get_dict_ru(dict_samp_stats, p)
    
    dict_samp_stats.to_excel(f'{output}.xlsx', index=False)
    
    return dict_samp_stats
    
        
        
        
        
    
    
    
    



In [ ]:
# --
list_res = []

# --
for p in files:
    with rio.open(os.path.join('spck_filt', p)) as src:
        samples_jun.to_crs(src.crs)
        mask_data, mask_transform = mask(dataset=src, shapes=samples_jun_reprojected.geometry, nodata=None, crop=True)
        #print(src.meta['crs'])
        x = mask_data
        x[0][x[0] == 0] = np.nan
        stats = resp_stats(x)
        list_res.append([p.split('_')[-1][:-4], stats['mean'], stats['std'], round(enl(x))])

In [4]:
def resp_stats(x):
    
    return {'mean': np.nanmean(x),
            'std': np.nanstd(x)}
    
def enl(x):
    return (resp_stats(x)['mean']/resp_stats(x)['std'])**2

In [7]:
from pathlib import Path

folder_path = Path('spck_filt')

# Filter the directory contents to include files only
files = [f.name for f in folder_path.iterdir()]

## Imagen completa

In [8]:
# --
list_res = []

# --
for p in files:
    with rio.open(os.path.join('spck_filt', p)) as src:
        x = src.read()
        x[0][x[0] == 0] = np.nan
        stats = resp_stats(x)
        list_res.append([p.split('_')[-1][:-4], stats['mean'], stats['std'], enl(x)])

In [9]:
# NOTA: Los resultados no tiene sentido porque lo estamos haciendo sobre toda la imagen. El resultado nos da como si
# estuvieramos viendo sobre menos de 1 look
list_res

[['SpkMn11',
  np.float32(0.108635224),
  np.float32(0.1279798),
  np.float32(0.7205407)],
 ['SpkMn3',
  np.float32(0.10866244),
  np.float32(0.17538741),
  np.float32(0.38385046)],
 ['original',
  np.float32(0.10866268),
  np.float32(0.22867459),
  np.float32(0.22580056)],
 ['SpkGamma3',
  np.float32(0.10711159),
  np.float32(0.22626284),
  np.float32(0.22410236)],
 ['SpkLee3',
  np.float32(0.10774692),
  np.float32(0.21965227),
  np.float32(0.24062367)],
 ['SpkGamma11',
  np.float32(0.107836954),
  np.float32(0.2280698),
  np.float32(0.22356288)],
 ['SpkLee11',
  np.float32(0.10842079),
  np.float32(0.22329915),
  np.float32(0.23574962)]]

## Muestras

### Juncal

In [10]:
# Read Juncal samples
samples_jun = gpd.read_file('Muestras/muestras_tp4_gk5_wgs84_juncal_1.shp')#.to_crs(epsg=9001)
samples_jun_reprojected = samples_jun.to_crs(src.crs)

# --
print(samples_jun.crs == src.crs)

True


In [103]:
'''# Visualizo junto a los polígonos de muestreo
with rio.open(os.path.join('spck_filt', files[0])) as src:
    img = src.read()
    extent = plotting_extent(src) 

# Reproyecto el archivo vectorial
samples_jun = samples_jun.to_crs(src.crs)

# Ploteamos la imagen con los vectores superpuestos
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img[0], extent = extent)
samples_jun.plot(ax=ax, legend = True)
ax.ticklabel_format(style='plain')

# Etiquetas y título
ax.set_ylabel('Latitud (°)')
ax.set_xlabel('Longitud (°)')

plt.show()'''

"# Visualizo junto a los polígonos de muestreo\nwith rio.open(os.path.join('spck_filt', files[0])) as src:\n    img = src.read()\n    extent = plotting_extent(src) \n\n# Reproyecto el archivo vectorial\nsamples_jun = samples_jun.to_crs(src.crs)\n\n# Ploteamos la imagen con los vectores superpuestos\nfig, ax = plt.subplots(figsize=(12, 8))\nax.imshow(img[0], extent = extent)\nsamples_jun.plot(ax=ax, legend = True)\nax.ticklabel_format(style='plain')\n\n# Etiquetas y título\nax.set_ylabel('Latitud (°)')\nax.set_xlabel('Longitud (°)')\n\nplt.show()"

In [11]:
# --
list_res = []

# --
for p in files:
    with rio.open(os.path.join('spck_filt', p)) as src:
        samples_jun.to_crs(src.crs)
        mask_data, mask_transform = mask(dataset=src, shapes=samples_jun_reprojected.geometry, nodata=None, crop=True)
        #print(src.meta['crs'])
        x = mask_data
        x[0][x[0] == 0] = np.nan
        stats = resp_stats(x)
        list_res.append([p.split('_')[-1][:-4], stats['mean'], stats['std'], round(enl(x))])


In [12]:
df_juncal = pd.DataFrame(list_res, columns = ['Imagen', 'Mean', 'Std', 'ENL'])

In [13]:
df_juncal

,Imagen,Mean,Std,ENL
0,SpkMn11,0.547760,0.048113,130
1,SpkMn3,0.572387,0.119431,23
2,original,0.573568,0.170289,11
3,SpkGamma3,0.565098,0.132107,18
4,SpkLee3,0.569417,0.128227,20
5,SpkGamma11,0.575376,0.156576,14
6,SpkLee11,0.565235,0.107571,28


In [ ]:
import numpy as np
from scipy.stats import gamma

def error_vs_probability(enl, minError_dB=0.1, maxError_dB=4, stepError_dB=0.01):
    """
    For a given ENL (l), compute probability for a range of error_dB values.

    Parameters
    ----------
    enl : float
        Equivalent Number of Looks (ENL), shape parameter of the Gamma pdf.
    minError_dB, maxError_dB, stepError_dB : float
        Range of error_dB values to evaluate.

    Returns
    -------
    error_dBs : ndarray
        The error_dB values evaluated.
    probabilities : ndarray
        The corresponding probability for each error_dB.
    """
    error_dBs = np.arange(minError_dB, maxError_dB + 1e-9, stepError_dB)

    lower = 10 ** (-error_dBs / 10)
    upper = 10 ** (error_dBs / 10)

    probabilities = gamma.cdf(upper, a=l, scale=1.0 / enl) - gamma.cdf(lower, a=enl, scale=1.0 / l)

    return error_dBs, probabilities

In [33]:
def get_unct(enl, p):
    # --
    error_dBs, probabilities = error_vs_probability(enl)
    
    # Get the index of differences (distance, not signed difference)
    idx = np.abs(probabilities - p).argmin() # Shortest distance means, that is the nearest probability value
    
    return error_dBs[idx]
    
    

In [34]:
# Lista de probabilidades
list_probs = [50, 90, 95]

for probs in list_probs:
    df_juncal[f'Inc_rad{probs}'] = df_juncal['ENL'].apply(lambda x: get_unct(x, probs/100))

In [35]:
df_juncal

,Imagen,Mean,Std,ENL,Inc_rad,Inc_rad50,Inc_rad90,Inc_rad95
0,SpkMn11,0.547760,0.048113,130,0.75,0.26,0.63,0.75
1,SpkMn3,0.572387,0.119431,23,1.81,0.61,1.51,1.81
2,original,0.573568,0.170289,11,2.66,0.89,2.21,2.66
3,SpkGamma3,0.565098,0.132107,18,2.05,0.69,1.71,2.05
4,SpkLee3,0.569417,0.128227,20,1.94,0.66,1.62,1.94
5,SpkGamma11,0.575376,0.156576,14,2.34,0.79,1.95,2.34
6,SpkLee11,0.565235,0.107571,28,1.63,0.56,1.36,1.63


### Bosque

In [110]:
# Read Bosque samples
samples_bosq = gpd.read_file('Muestras/muestras_tp4_gk5_wgs84_bosque_1.shp')#.to_crs(epsg=9001)
samples_bosq_reprojected = samples_bosq.to_crs(src.crs)

In [111]:
# --
list_res = []

# --
for p in files:
    with rio.open(os.path.join('spck_filt', p)) as src:
        samples_jun.to_crs(src.crs)
        mask_data, mask_transform = mask(dataset=src, shapes=samples_bosq_reprojected.geometry, nodata=None, crop=True)
        #print(src.meta['crs'])
        x = mask_data
        x[0][x[0] == 0] = np.nan
        stats = resp_stats(x)
        list_res.append([p.split('_')[-1][:-4], stats['mean'], stats['std'], round(enl(x))])

In [112]:
df_bosque = pd.DataFrame(list_res, columns = ['Imagen', 'Mean', 'Std', 'ENL'])
df_bosque

,Imagen,Mean,Std,ENL
0,SpkMn11,0.090204,0.003894,537
1,SpkMn3,0.090521,0.014368,40
2,SpkGamma3,0.088936,0.018371,23
3,SpkLee3,0.089914,0.017085,28
4,sub,0.090585,0.026188,12
5,SpkGamma11,0.088048,0.011379,60
6,SpkLee11,0.090390,0.010835,70


In [114]:
# Read Agua samples
samples_lag = gpd.read_file('Muestras/muestras_tp4_gk5_wgs84_lagunas_1.shp')#.to_crs(epsg=9001)
samples_lag_reprojected = samples_lag.to_crs(src.crs)

In [115]:
# --
list_res = []

# --
for p in files:
    with rio.open(os.path.join('spck_filt', p)) as src:
        samples_jun.to_crs(src.crs)
        mask_data, mask_transform = mask(dataset=src, shapes=samples_lag_reprojected.geometry, nodata=None, crop=True)
        #print(src.meta['crs'])
        x = mask_data
        x[0][x[0] == 0] = np.nan
        stats = resp_stats(x)
        list_res.append([p.split('_')[-1][:-4], stats['mean'], stats['std'], round(enl(x))])

In [116]:
df_lag = pd.DataFrame(list_res, columns = ['Imagen', 'Mean', 'Std', 'ENL'])
df_lag

,Imagen,Mean,Std,ENL
0,SpkMn11,0.005133,0.000180,815
1,SpkMn3,0.005224,0.000595,77
2,SpkGamma3,0.005187,0.000664,61
3,SpkLee3,0.005209,0.000639,66
4,sub,0.005257,0.001083,24
5,SpkGamma11,0.005134,0.000184,775
6,SpkLee11,0.005140,0.000180,813
